# Synthetic dataset training run

# Continual Learning â€” Colab Training Notebook

**Before running:**
1. Set runtime to **GPU** (Runtime â†’ Change runtime type â†’ T4 GPU or better)
2. Edit the **Configuration** cell below
3. Run all cells top-to-bottom

Checkpoints and the dataset cache are stored on **Google Drive** and survive session restarts.
If a checkpoint already exists for your `RUN_ID`, training will automatically resume from it.

## Configuration
Edit the values below before running the notebook.

In [5]:
# ============================================================
#  CONFIGURE EVERYTHING HERE
# ============================================================

# Training method â€” choose one:
#   "full_ft"  â€” fine-tune all parameters (catastrophic forgetting baseline)
#   "lora"     â€” parameter-efficient LoRA adapters
#   "smf"      â€” frozen backbone + sparse trainable memory
#   "casm"     â€” versioned memory bank with contradiction-aware routing
METHOD = "casm"

# HuggingFace model (must have approved access)
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Unique experiment name â€” checkpoints are stored under this ID
RUN_ID = "llama1b_synthetic_casm_period_slots_1"

# Periods to train â€” full sequence is all four in order
# Each period is one temporal snapshot of the synthetic dataset
PERIODS = ["2018", "2020", "2022", "2024"]  # full run: ["2018", "2020", "2022", "2024"]

# Google Drive folder â€” checkpoints and dataset cache go here
DRIVE_DIR = "/content/drive/MyDrive/continual_learning/synthetic"

# ---- Dataset fraction ----
# Fraction of the dataset to use for both training and evaluation.
# The fraction is applied proportionally and independently to the changed and
# unchanged probe splits, so training and evaluation always cover the same
# examples.  None means use all data; 0.1 means use 10% of each split.
DATASET_FRACTION = .5  # synthetic dataset is small (605 facts) â€” use all data

# ---- Core hyperparameters ----
BATCH_SIZE              = 1
GRAD_ACCUM_STEPS        = 4       # effective batch size = 4
LEARNING_RATE           = 5e-4
EPOCHS_PER_PERIOD       = 20     # passes over all passages
MAX_PASSAGES_PER_PERIOD = None    # use all available passages
PRECISION               = "bfloat16"
SEED                    = 42

# ---- Activation capture ----
# Records per-layer hidden-state activations after each period.
# Output saved to periods/<period>/activations.pt and activation_metadata.json
# alongside eval outputs â€” copy to another machine for plotting, no GPU needed.
CAPTURE_ACTIVATIONS = True

# ---- LoRA settings (only used when METHOD == "lora") ----
LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ---- SMF settings (only used when METHOD == "smf") ----
# Llama-3.2-1B has 16 transformer layers (indices 0-15)
SMF_MEMORY_SIZE            = 64
SMF_SPARSITY_RATIO         = 0.1
SMF_UPDATE_LAYERS          = [4, 8, 12]  # mid-to-late layers
SMF_REGULARIZATION_WEIGHT  = 0.01
SMF_LEARNING_RATE          = 1e-3

# ---- CASM settings (only used when METHOD == "casm") ----
CASM_SLOTS_PER_PERIOD       = 8   # 8 slots x 4 periods = 32 total; must equal CASM_NUM_SLOTS / len(PERIODS)
CASM_NUM_SLOTS              = 32  # must equal CASM_SLOTS_PER_PERIOD * len(PERIODS)
CASM_ROUTER_HIDDEN_SIZE     = 256
CASM_TOP_K                  = 2
CASM_ROUTER_TEMPERATURE     = 1.0
CASM_SPARSITY_WEIGHT        = 0.01
CASM_OVERLAP_WEIGHT         = 0.01
CASM_BRANCH_ON_CONTRADICTION = False  # must be False with period-deterministic slots
CASM_MEMORY_SIZE            = 64
CASM_ROUTER_TYPE            = 'similarity'  # 'mlp' = CASMRouter (learned, default), 'similarity' = SimilarityRouter (cosine, no training)

## Setup

In [6]:
# Mount Google Drive for persistent checkpoint and dataset storage
from google.colab import drive
drive.mount("/content/drive")

import os

CHECKPOINT_DIR   = os.path.join(DRIVE_DIR, "checkpoints")
DATASET_CACHE_DIR = os.path.join(DRIVE_DIR, "dataset_cache")
REPO_DIR         = "/content/Continual-Learning"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATASET_CACHE_DIR, exist_ok=True)

print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print(f"Dataset cache:   {DATASET_CACHE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint dir:  /content/drive/MyDrive/continual_learning/synthetic/checkpoints
Dataset cache:   /content/drive/MyDrive/continual_learning/synthetic/dataset_cache


In [7]:
# Clone the repo (or pull if already cloned this session)
import subprocess

REPO_URL = "https://github.com/athirai-s/Continual-Learning"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned â€” pulling latest changes...")
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "pull"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print(f"Cloning {REPO_URL} ...")
    result = subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{result.stderr}")

print("Repo ready.")

Cloning https://github.com/athirai-s/Continual-Learning ...
Cloning into '/content/Continual-Learning'...
Repo ready.


In [8]:
# Install dependencies not pre-installed on Colab
# transformers 5.x is required â€” Colab ships with 4.x by default
!pip install -q "transformers>=5.3.0" "peft>=0.18.1" "trl>=0.29.0" "datasets>=4.6.1" "sentence-transformers>=3.0.0"

# Add the repo to the Python path so imports work
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Dependencies installed. Repo on path.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 163.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.0 MB/s eta 0:00:00
Dependencies installed. Repo on path.


## Dataset

Training passages come from `data/augmented/synthetic/<period>.csv` â€” these are
committed directly in the repo, so no download is needed.

Evaluation probes come from `data/probes.json`, which is also committed in the repo.
`SyntheticDataset` loads this file directly â€” no zip extraction needed.
The synthetic dataset contains 605 facts with ~150 changed probes per period boundary.

In [9]:
import os

# Verify augmented training CSVs (data/augmented/synthetic/<period>.csv)
aug_dir = os.path.join(REPO_DIR, "data", "augmented", "synthetic")
for period in PERIODS:
    csv_path = os.path.join(aug_dir, f"{period}.csv")
    assert os.path.exists(csv_path), f"Synthetic CSV missing: {csv_path}"
    print(f"  {period}.csv: OK ({os.path.getsize(csv_path):,} bytes)")

# Verify probes file
probes_json = os.path.join(REPO_DIR, "data", "probes.json")
assert os.path.exists(probes_json), (
    f"probes.json not found at {probes_json}. "
    "Run data/build_probes.py to generate it."
)
print(f"probes.json: OK ({os.path.getsize(probes_json):,} bytes)")
print("All synthetic dataset files present.")


  2018.csv: OK (73,841 bytes)
  2020.csv: OK (74,448 bytes)
  2022.csv: OK (74,093 bytes)
  2024.csv: OK (74,281 bytes)
probes.json: OK (1,247,972 bytes)
All synthetic dataset files present.


## HuggingFace Authentication

Required for gated models like Llama. Log in with a token that has **read** access to
`meta-llama/Llama-3.2-1B`. You can create a token at https://huggingface.co/settings/tokens

In [10]:
from huggingface_hub import login
login()  # will prompt for your HuggingFace access token

## Training

In [11]:
import os

aug_dir = os.path.join(REPO_DIR, "data", "augmented", "synthetic")
all_ok = True
for period in PERIODS:
    path = os.path.join(aug_dir, f"{period}.csv")
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  {period}.csv: {status} ({size:,} bytes)")
    if not exists:
        all_ok = False

if all_ok:
    print("All synthetic CSVs found â€” ready to train.")
else:
    raise FileNotFoundError(
        "One or more synthetic CSVs are missing. "
        "Check that data/augmented/synthetic/<period>.csv files are committed to the repo."
    )


  2018.csv: OK (73,841 bytes)
  2020.csv: OK (74,448 bytes)
  2022.csv: OK (74,093 bytes)
  2024.csv: OK (74,281 bytes)
All synthetic CSVs found â€” ready to train.


In [12]:
# Build TrainConfig from configuration variables set at the top
from training.train_config import TrainConfig

config_kwargs = dict(
    run_id=RUN_ID,
    model_name=MODEL_NAME,
    method=METHOD,
    dataset_name="synthetic",
    precision=PRECISION,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    epochs_per_period=EPOCHS_PER_PERIOD,
    max_passages_per_period=MAX_PASSAGES_PER_PERIOD,
    dataset_fraction=DATASET_FRACTION,
    log_every_n_steps=10,
    eval_after_each_period=True,
    capture_activations=CAPTURE_ACTIVATIONS,
    seed=SEED,
)

if METHOD == "lora":
    config_kwargs.update(
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        lora_target_modules=LORA_TARGET_MODULES,
    )
elif METHOD == "smf":
    config_kwargs.update(
        smf_memory_size=SMF_MEMORY_SIZE,
        smf_sparsity_ratio=SMF_SPARSITY_RATIO,
        smf_update_layers=SMF_UPDATE_LAYERS,
        smf_regularization_weight=SMF_REGULARIZATION_WEIGHT,
        smf_learning_rate=SMF_LEARNING_RATE,
        smf_freeze_backbone=True,
    )
elif METHOD == "casm":
    config_kwargs.update(
        casm_num_slots=CASM_NUM_SLOTS,
        casm_router_hidden_size=CASM_ROUTER_HIDDEN_SIZE,
        casm_top_k=CASM_TOP_K,
        casm_router_temperature=CASM_ROUTER_TEMPERATURE,
        casm_sparsity_weight=CASM_SPARSITY_WEIGHT,
        casm_overlap_weight=CASM_OVERLAP_WEIGHT,
        casm_branch_on_contradiction=CASM_BRANCH_ON_CONTRADICTION,
        casm_memory_size=CASM_MEMORY_SIZE,
        casm_router_type=CASM_ROUTER_TYPE,
        casm_slots_per_period=CASM_SLOTS_PER_PERIOD,
    )

cfg = TrainConfig(**config_kwargs)
cfg.validate()

print(f"Method:               {cfg.method}")
print(f"Model:                {cfg.model_name}")
print(f"Periods:              {PERIODS}")
print(f"Batch size:           {cfg.batch_size}")
print(f"Grad accum steps:     {cfg.grad_accum_steps}  (eff. batch = {cfg.batch_size * cfg.grad_accum_steps})")
print(f"Learning rate:        {cfg.learning_rate}")
print(f"Epochs per period:    {cfg.epochs_per_period}")
print(f"Max passages:         {cfg.max_passages_per_period}")
print(f"Dataset fraction:     {cfg.dataset_fraction if cfg.dataset_fraction is not None else 'all (None)'}")
print(f"Capture activations:  {cfg.capture_activations}")
print(f"Checkpoint dir:       {CHECKPOINT_DIR}")

Method:               casm
Model:                meta-llama/Llama-3.2-1B
Periods:              ['2018', '2020', '2022', '2024']
Batch size:           1
Grad accum steps:     4  (eff. batch = 4)
Learning rate:        0.0005
Epochs per period:    20
Max passages:         None
Dataset fraction:     0.5
Capture activations:  True
Checkpoint dir:       /content/drive/MyDrive/continual_learning/synthetic/checkpoints


In [13]:
# Detect existing checkpoint for automatic resume
import json

RESUME_FROM = None
run_root = os.path.join(CHECKPOINT_DIR, RUN_ID)
latest_json = os.path.join(run_root, "latest.json")

if os.path.exists(latest_json):
    with open(latest_json) as f:
        pointer = json.load(f)
    last_period = pointer.get("last_period", "unknown")
    print(f"Existing checkpoint found (last period: {last_period}).")
    print(f"Training will resume from: {run_root}")
    RESUME_FROM = run_root
else:
    print("No existing checkpoint â€” starting fresh.")

No existing checkpoint â€” starting fresh.


In [14]:
# ── Data verification ─────────────────────────────────────────────────────
# Loads each period using the same factory as training — augmented CSVs,
# same dataset_fraction — so counts here match what training will actually see.
from training.train_runner import build_synthetic_dataset

print(f"Method: {cfg.method}")
print(f"Dataset: {cfg.dataset_name}")
print(f"Periods: {PERIODS}")
print(f"Dataset fraction: {cfg.dataset_fraction if cfg.dataset_fraction is not None else 'all (None)'}")
print()

for period in PERIODS:
    ds = build_synthetic_dataset(period, cfg)
    ds.load("changed")
    ds.load("unchanged")

    passages   = ds.get_train_passages()
    changed    = ds.get_probes("changed")
    unchanged  = ds.get_probes("unchanged")

    print(f"Period: {period}")
    print(f"  Training passages : {len(passages)}")
    print(f"  Changed probes    : {len(changed)}")
    print(f"  Unchanged probes  : {len(unchanged)}")

    if passages:
        print(f"  Sample passage    : {passages[0][:120]!r}")
    if changed:
        p = changed[0]
        print(f"  Sample changed    : [{p.subject}] {p.relation} -> {p.current_value!r}  (was {p.previous_value!r})")
    if unchanged:
        p = unchanged[0]
        print(f"  Sample unchanged  : [{p.subject}] {p.relation} -> {p.current_value!r}")
    print()

print("Data looks correct — proceed to training.")


Method: casm
Dataset: synthetic
Periods: ['2018', '2020', '2022', '2024']
Dataset fraction: 0.5

  dataset_fraction=0.500: 303/605 changed probes, 0/0 unchanged probes, 303/605 training passages
Period: 2018
  Training passages : 303
  Changed probes    : 303
  Unchanged probes  : 0
  Sample passage    : '[2018] The primary berth depth of Azure Tide Harbor is 12 fathoms. Ships can dock safely there. It accommodates large ve'
  Sample changed    : [Azure Tide Harbor] primary berth depth -> '12 fathoms'  (was None)

  dataset_fraction=0.500: 75/150 changed probes, 228/455 unchanged probes, 303/605 training passages
Period: 2020
  Training passages : 303
  Changed probes    : 75
  Unchanged probes  : 228
  Sample passage    : '[2020] The primary berth depth of Azure Tide Harbor is 12 fathoms. Ships can dock safely. It is a busy port.'
  Sample changed    : [Misty Fjord Logistics] dock capacity -> '25 ships'  (was None)
  Sample unchanged  : [Azure Tide Harbor] primary berth depth -> '12 f

In [15]:
from training.train_runner import (
    run_training,
    build_real_model_and_tokenizer,
    load_real_model_and_tokenizer,
    build_synthetic_dataset,
)

dataset_factory = build_synthetic_dataset
print("Dataset factory: synthetic CSVs (data/augmented/synthetic/)")
results = run_training(
    cfg,
    model_factory=build_real_model_and_tokenizer,
    resume_model_factory=load_real_model_and_tokenizer,
    dataset_factory=dataset_factory,
    checkpoint_dir=CHECKPOINT_DIR,
    training_units=PERIODS,
    resume_from=RESUME_FROM,
)

print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    loss = r.get("train_loss_final")
    print(f"  Final loss:        {loss:.4f}" if loss is not None else "  Final loss:        N/A")
    print(f"  Passages trained:  {r.get('n_passages_trained', 'N/A')}")
    print(f"  Train time (s):    {r.get('train_duration_sec', 0):.1f}")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            if isinstance(eval_result, dict):
                plasticity = eval_result.get("plasticity")
                stability  = eval_result.get("stability")
                token_f1   = eval_result.get("token_f1")
            else:
                plasticity = eval_result.plasticity
                stability  = eval_result.stability
                token_f1   = eval_result.token_f1
            p = f"{plasticity:.3f}" if plasticity is not None else "N/A"
            s = f"{stability:.3f}"  if stability  is not None else "N/A"
            f = f"{token_f1:.3f}"   if token_f1   is not None else "N/A"
            print(f"  Eval [{split:9s}]: plasticity={p}  stability={s}  token_f1={f}")
    print(f"  Checkpoint:        {r.get('checkpoint_path', 'N/A')}")

Dataset factory: synthetic CSVs (data/augmented/synthetic/)
Training config:
TrainConfig(model_name='meta-llama/Llama-3.2-1B', method='casm', dataset_name='synthetic', precision='bfloat16', learning_rate=0.0005, batch_size=1, grad_accum_steps=4, epochs_per_period=20, grad_clip=1.0, warmup_steps=100, min_passage_length=0, deduplicate=True, weighted_sampling=False, max_passages_per_period=None, dataset_fraction=0.5, shuffle_by_relation=True, contradiction_batch_frac=0.25, run_id='llama1b_synthetic_casm_similarity_12', log_every_n_steps=10, eval_after_each_period=True, capture_activations=True, checkpoint_dir='/content/drive/MyDrive/continual_learning/synthetic/checkpoints', checkpoint_every_n_optimizer_steps=None, seed=42, lora_r=None, lora_alpha=None, lora_dropout=0.05, lora_target_modules=None, smf_memory_size=None, smf_sparsity_ratio=None, smf_update_layers=None, smf_regularization_weight=0.01, smf_freeze_backbone=True, smf_learning_rate=None, casm_num_slots=8, casm_router_hidden_size

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

  dataset_fraction=0.500: 75/150 changed probes, 228/455 unchanged probes, 303/605 training passages
Activations saved: 20 probes × 16 layers → /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_12/periods/2020/activations.pt
Training result:
  Final loss: 3.898375153541565
  Passages trained: 6060
  Contradiction passages: 70
  Train duration (sec): 281.15
Checkpoint saved to: /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_12/checkpoints/ckpt-000002

=== Training unit: 2022 ===
  dataset_fraction=0.500: 75/149 changed probes, 228/456 unchanged probes, 303/605 training passages
  Passages:      6060 (20 epoch(s))
  Optimizer steps: 1515  |  eff_batch=4  |  lr=0.00e+00
  Device:          cuda
  step   10/1515 (  0.7%)  loss=11.4093  lr=5.00e-05    451 tok/s  elapsed=00:02  ETA=07:26
  step   20/1515 (  1.3%)  loss=10.4259  lr=1.00e-04    465 tok/s  elapsed=00:05  ETA=07:06
  step   30/1515 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  dataset_fraction=0.500: 75/149 changed probes, 228/456 unchanged probes, 303/605 training passages
Activations saved: 20 probes × 16 layers → /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_12/periods/2022/activations.pt
Training result:
  Final loss: 6.015248775482178
  Passages trained: 6060
  Contradiction passages: 75
  Train duration (sec): 404.09
Checkpoint saved to: /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_12/checkpoints/ckpt-000003

=== Training unit: 2024 ===
  dataset_fraction=0.500: 75/150 changed probes, 228/455 unchanged probes, 303/605 training passages
  Passages:      6060 (20 epoch(s))
  Optimizer steps: 1515  |  eff_batch=4  |  lr=0.00e+00
  Device:          cuda
  step   10/1515 (  0.7%)  loss=13.2664  lr=5.00e-05    347 tok/s  elapsed=00:04  ETA=10:04
  step   20/1515 (  1.3%)  loss=12.3713  lr=1.00e-04    313 tok/s  elapsed=00:07  ETA=09:32
  step   30/1515 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  dataset_fraction=0.500: 75/150 changed probes, 228/455 unchanged probes, 303/605 training passages
Activations saved: 20 probes × 16 layers → /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_12/periods/2024/activations.pt
Training result:
  Final loss: 7.444415092468262
  Passages trained: 6060
  Contradiction passages: 75
  Train duration (sec): 532.32
Checkpoint saved to: /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_12/checkpoints/ckpt-000004

Done.

TRAINING SUMMARY

2018:
  Final loss:        2.0590
  Passages trained:  6060
  Train time (s):    168.5
  Eval [changed  ]: plasticity=0.000  stability=0.020  token_f1=0.010
  Eval [unchanged]: plasticity=0.000  stability=0.000  token_f1=0.000
  Checkpoint:        /content/drive/MyDrive/continual_learning/synthetic/checkpoints/llama1b_synthetic_casm_similarity_12/checkpoints/ckpt-000001

2020:
  Final loss:        3.8984
  Passages tr

In [16]:
# Print eval summary from the completed run (no retraining needed)
print("\n" + "="*50)
print("EVAL SUMMARY")
print("="*50)
for i, r in enumerate(results):
    period = PERIODS[i] if i < len(PERIODS) else f"period_{i}"
    print(f"\n{period}:")
    if "evaluation" in r:
        for split, eval_result in r["evaluation"].items():
            if isinstance(eval_result, dict):
                plasticity = eval_result.get("plasticity")
                stability  = eval_result.get("stability")
                token_f1   = eval_result.get("token_f1")
            else:
                plasticity = eval_result.plasticity
                stability  = eval_result.stability
                token_f1   = eval_result.token_f1
            p = f"{plasticity:.3f}" if plasticity is not None else "N/A"
            s = f"{stability:.3f}"  if stability  is not None else "N/A"
            f = f"{token_f1:.3f}"   if token_f1   is not None else "N/A"
            print(f"  [{split:9s}]: plasticity={p}  stability={s}  token_f1={f}")
    else:
        print("  (no evaluation results)")


EVAL SUMMARY

2018:
  [changed  ]: plasticity=0.000  stability=0.020  token_f1=0.010
  [unchanged]: plasticity=0.000  stability=0.000  token_f1=0.000

2020:
  [changed  ]: plasticity=0.000  stability=0.000  token_f1=0.012
  [unchanged]: plasticity=0.000  stability=0.026  token_f1=0.012

2022:
  [changed  ]: plasticity=0.013  stability=0.000  token_f1=0.000
  [unchanged]: plasticity=0.000  stability=0.022  token_f1=0.010

2024:
  [changed  ]: plasticity=0.000  stability=0.000  token_f1=0.006
  [unchanged]: plasticity=0.000  stability=0.018  token_f1=0.016


In [17]:
import torch
from training.train_runner import load_real_model_and_tokenizer

checkpoint_path = results[0]["checkpoint_path"]
model, tokenizer = load_real_model_and_tokenizer(cfg, checkpoint_path)
model.eval()

device = next(model.parameters()).device

sample_dataset = dataset_factory(PERIODS[-1], cfg)
sample_dataset.load("changed")
probes = sample_dataset.get_probes("changed")[:5]

for probe in probes:
    inputs = tokenizer(probe.prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
        )
    generated = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print(f"Prompt:   {probe.prompt}")
    print(f"Expected: {probe.ground_truth}")
    print(f"Got:      {generated!r}")
    print()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  dataset_fraction=0.500: 75/150 changed probes, 228/455 unchanged probes, 303/605 training passages


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Prompt:   [2024] The chief engineer of Seabridge Construction is
Expected: Lorien Vance
Got:      'a 50-year-old'



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   [2024] The navigational hazard level of Whispering Shoals is
Expected: low
Got:      '1.5.'



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt:   [2024] The radar frequency of Port Lumina Control is
Expected: S-band
Got:      '2.4 GHz'

Prompt:   [2024] The current direction of Serpent's Coil Strait is
Expected: southwest
Got:      'to the north. The'



In [18]:
from collections import defaultdict

def param_summary(model):
    total_params = 0
    trainable_params = 0
    groups = defaultdict(lambda: {"total": 0, "trainable": 0})

    for name, param in model.named_parameters():
        n = param.numel()
        top = name.split(".")[0]
        total_params += n
        groups[top]["total"] += n
        if param.requires_grad:
            trainable_params += n
            groups[top]["trainable"] += n

    col = 42
    print(f"\n{'Module':<{col}} {'Total params':>14} {'Trainable':>12} {'%':>7}")
    print("-" * (col + 36))
    for group, counts in sorted(groups.items()):
        t, tr = counts["total"], counts["trainable"]
        pct = 100 * tr / t if t else 0
        marker = "  <-- updated" if tr > 0 else ""
        print(f"  {group:<{col-2}} {t:>14,} {tr:>12,} {pct:>6.1f}%{marker}")
    print("-" * (col + 36))
    pct = 100 * trainable_params / total_params if total_params else 0
    print(f"  {'TOTAL':<{col-2}} {total_params:>14,} {trainable_params:>12,} {pct:>6.1f}%")
    print()
    print(f"Trainable: {trainable_params:,} / {total_params:,}  ({pct:.4f}% of model)")

param_summary(model)


Module                                       Total params    Trainable       %
------------------------------------------------------------------------------
  backbone                                  1,235,814,400            0    0.0%
  slot_bank                                     2,097,664    2,097,664  100.0%  <-- updated
------------------------------------------------------------------------------
  TOTAL                                     1,237,912,064    2,097,664    0.2%

Trainable: 2,097,664 / 1,237,912,064  (0.1695% of model)
